<a href="https://colab.research.google.com/github/AmitabhDey-byte/AmitabhDey-byte/blob/main/Stock_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [179]:
import pandas as pd
import numpy as np
import yfinance as yf
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
df = pd.read_csv('sample_data/stock_news_2016 to 2026.csv')

In [180]:
df.head()

,date,title,description,url,source_file,categories,matched_keywords,relevance_score,has_negation,impact_tier
0,2016-01-01,Lending to foreign step-down arms of Indian fi...,RBI slaps 2% additional provision for such loans,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,rbi,2,False,LOW
1,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2,False,LOW
2,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3,False,MEDIUM
3,2016-01-02,SBI: Lending rate cut unlikely till end-Mar,Country's largest lender SBI today ruled out f...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,stock_specific,sbi,2,True,LOW
4,2016-01-03,"ASK Group to invest Rs 1,500 cr in real estate...",ASK Group plans to step up its equity investme...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_cement_infra,real estate,1,False,LOW


In [181]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139919 entries, 0 to 139918
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   date              139919 non-null  object
 1   title             139919 non-null  object
 2   description       139919 non-null  object
 3   url               139919 non-null  object
 4   source_file       139919 non-null  object
 5   categories        139919 non-null  object
 6   matched_keywords  139919 non-null  object
 7   relevance_score   139919 non-null  int64 
 8   has_negation      139919 non-null  bool  
 9   impact_tier       139919 non-null  object
dtypes: bool(1), int64(1), object(8)
memory usage: 9.7+ MB


In [182]:
df = df.dropna()
len(df)

139919

In [183]:
#ok so no missing rows

In [184]:
#i need stock market data to merge
#so using yfinance

In [185]:
df2 = yf.download("^NSEI", start="2016-01-01", end="2026-01-01")

/tmp/ipykernel_828/1582892846.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df2 = yf.download("^NSEI", start="2016-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed


In [186]:
df2.head()

Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700
2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100
2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900
2016-01-08,7601.350098,7634.100098,7581.049805,7611.649902,157400


In [187]:
df2.info

<bound method DataFrame.info of Price              Close          High           Low          Open  Volume
Ticker             ^NSEI         ^NSEI         ^NSEI         ^NSEI   ^NSEI
Date                                                                      
2016-01-04   7791.299805   7937.549805   7781.100098   7924.549805  134700
2016-01-05   7784.649902   7831.200195   7763.250000   7828.399902  145200
2016-01-06   7741.000000   7800.950195   7721.200195   7788.049805  147100
2016-01-07   7568.299805   7674.950195   7556.600098   7673.350098  188900
2016-01-08   7601.350098   7634.100098   7581.049805   7611.649902  157400
...                  ...           ...           ...           ...     ...
2025-12-24  26142.099609  26236.400391  26123.000000  26170.650391  188800
2025-12-26  26042.300781  26144.199219  26008.599609  26121.250000  142200
2025-12-29  25942.099609  26106.800781  25920.300781  26063.349609  234300
2025-12-30  25938.849609  25976.750000  25878.000000  25940.900391  396900
2025-12-31  26129.599609  26187.949219  25969.000000  25971.050781  246300

[2464 rows x 5 columns]>

In [188]:
df['date'] = pd.to_datetime(df['date']).dt.date

In [189]:
df['news_data'] = df['title'].astype(str) + " " + df['description'].astype(str)

print(df['news_data'].head(1))

0    Lending to foreign step-down arms of Indian fi...
Name: news_data, dtype: object


In [190]:
df2.columns = [col[0] if isinstance(col,  tuple) else col for col in df2.columns]

In [191]:
df2.reset_index(inplace=True)
df2.rename(columns={"Price Ticker Date": "date"}, inplace=True)
df2['date'] = pd.to_datetime(df['date']).dt.date

In [192]:
df2.head()

,Date,Close,High,Low,Open,Volume,date
0,2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700,2016-01-01
1,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200,2016-01-02
2,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100,2016-01-02
3,2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900,2016-01-02
4,2016-01-08,7601.350098,7634.100098,7581.049805,7611.649902,157400,2016-01-03


In [193]:
df3 = pd.merge(df, df2, on='date')

In [194]:
df3.head(6)

,date,title,description,url,source_file,categories,matched_keywords,relevance_score,has_negation,impact_tier,news_data,Date,Close,High,Low,Open,Volume
0,2016-01-01,Lending to foreign step-down arms of Indian fi...,RBI slaps 2% additional provision for such loans,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,rbi,2,False,LOW,Lending to foreign step-down arms of Indian fi...,2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700
1,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
2,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100
3,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900
4,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3,False,MEDIUM,Forex reserves up $943 mn India's foreign exch...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
5,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3,False,MEDIUM,Forex reserves up $943 mn India's foreign exch...,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100


In [195]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64886 entries, 0 to 64885
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              64886 non-null  object        
 1   title             64886 non-null  object        
 2   description       64886 non-null  object        
 3   url               64886 non-null  object        
 4   source_file       64886 non-null  object        
 5   categories        64886 non-null  object        
 6   matched_keywords  64886 non-null  object        
 7   relevance_score   64886 non-null  int64         
 8   has_negation      64886 non-null  bool          
 9   impact_tier       64886 non-null  object        
 10  news_data         64886 non-null  object        
 11  Date              64886 non-null  datetime64[ns]
 12  Close             64886 non-null  float64       
 13  High              64886 non-null  float64       
 14  Low               6488

In [196]:
df3 = df3.drop(['date','title','description','url','source_file'], axis=1)

In [197]:
df3.head()

,categories,matched_keywords,relevance_score,has_negation,impact_tier,news_data,Date,Close,High,Low,Open,Volume
0,sector_banking_finance,rbi,2,False,LOW,Lending to foreign step-down arms of Indian fi...,2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700
1,sector_banking_finance,basel,2,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
2,sector_banking_finance,basel,2,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100
3,sector_banking_finance,basel,2,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900
4,macro_government,forex reserve,3,False,MEDIUM,Forex reserves up $943 mn India's foreign exch...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200


In [198]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64886 entries, 0 to 64885
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   categories        64886 non-null  object        
 1   matched_keywords  64886 non-null  object        
 2   relevance_score   64886 non-null  int64         
 3   has_negation      64886 non-null  bool          
 4   impact_tier       64886 non-null  object        
 5   news_data         64886 non-null  object        
 6   Date              64886 non-null  datetime64[ns]
 7   Close             64886 non-null  float64       
 8   High              64886 non-null  float64       
 9   Low               64886 non-null  float64       
 10  Open              64886 non-null  float64       
 11  Volume            64886 non-null  int64         
dtypes: bool(1), datetime64[ns](1), float64(4), int64(2), object(4)
memory usage: 5.5+ MB


In [199]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

In [200]:
df3['sentiment'] = df3['news_data'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

In [201]:
df3[['news_data', 'sentiment']].head(20)

,news_data,sentiment
0,Lending to foreign step-down arms of Indian fi...,0.0000
1,IDBI Bank to raise Rs 900 cr through Basel-III...,0.0000
2,IDBI Bank to raise Rs 900 cr through Basel-III...,0.0000
3,IDBI Bank to raise Rs 900 cr through Basel-III...,0.0000
4,Forex reserves up $943 mn India's foreign exch...,0.0000
5,Forex reserves up $943 mn India's foreign exch...,0.0000
6,Forex reserves up $943 mn India's foreign exch...,0.0000
7,SBI: Lending rate cut unlikely till end-Mar Co...,-0.4939
8,SBI: Lending rate cut unlikely till end-Mar Co...,-0.4939
9,SBI: Lending rate cut unlikely till end-Mar Co...,-0.4939


In [202]:
# 5 columns removed which were unnecessary to remove noise

In [203]:
df3['target'] = df3['Close'].shift(-1)

In [204]:
df3= df3.dropna()

In [205]:
X = df3[['Close','High','Low','Open','Volume','sentiment']]
Y =df3['target']

In [206]:
X_train, X_test, y_train, y_test = train_test_split(X, Y,test_size=0.2,shuffle=False)

In [207]:
model = XGBRegressor(n_estimators=300,learning_rate=0.05,max_depth=4,
                      subsample=0.8,colsample_bytree=0.8,
                       reg_alpha=1,reg_lambda=1,
                     tree_method='hist',device='cuda')

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [208]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

In [209]:
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
print("Train R2:", train_r2)
print("Test R2 :", test_r2)
print("Train RMSE:", train_rmse)
print("Test RMSE :", test_rmse)

Train R2: 0.9989992506513778
Test R2 : 0.9237272489318917
Train RMSE: 142.093739804397
Test RMSE : 249.44690713109165


there is a bit or mild overlifting....but fixable..all left to do is apply sentimen analysis and then grid search